In [ ]:
"""
Concepts:

1. Tree Structure
2. Feature Importance
3. Overfitting
4. Pruning
5. Tree Visualization
6. Max Depth
7. Min Samples Split
8. Min Samples Leaf
"""

# =====================================================
# IMPORTS
# =====================================================

import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import (train_test_split,cross_val_score,GridSearchCV,RandomizedSearchCV)
from sklearn.tree import (DecisionTreeRegressor,plot_tree)
from sklearn.metrics import (mean_absolute_error,mean_squared_error,r2_score)

In [ ]:
# =====================================================
# STEP 1 : DATA UNDERSTANDING
# =====================================================

housing = fetch_california_housing()
df = pd.DataFrame(housing.data,columns=housing.feature_names)
df["Target"] = housing.target

print(df.head())
print(df.info())
print(df.describe())


In [ ]:
# =====================================================
# STEP 2 : EDA
# =====================================================

print(df.isnull().sum())
print("Duplicates:", df.duplicated().sum())
print(df.corr())


In [ ]:
# =====================================================
# STEP 3 : TRAIN / VALIDATION / TEST
# =====================================================

X = df.drop("Target", axis=1)
y = df["Target"]
X_temp, X_test, y_temp, y_test = train_test_split(X,y,test_size=0.15,random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp,y_temp,test_size=0.1765,random_state=42)

# =====================================================
# STEP 4 : PREPROCESSING
# =====================================================

# Trees DO NOT require scaling.
# Because they split based on thresholds.

In [ ]:
# =====================================================
# STEP 5 : BASELINE MODEL
# =====================================================

dtr = DecisionTreeRegressor(random_state=42)
dtr.fit(X_train,y_train)

In [ ]:
# =====================================================
# STEP 6 : VALIDATION METRICS with base model
# =====================================================

y_val_pred = dtr.predict(X_val)
mae = mean_absolute_error(y_val,y_val_pred)
rmse = np.sqrt(mean_squared_error(y_val,y_val_pred))
r2 = r2_score(y_val,y_val_pred)

print("\nBASELINE TREE")
print("MAE :", mae)
print("RMSE:", rmse)
print("R2 :", r2)

In [ ]:
# =====================================================
# STEP 7 : CROSS VALIDATION
# =====================================================

cv_scores = cross_val_score(dtr,X_train,y_train,cv=5,scoring="r2",n_jobs=-1)
print("\nCV Mean:", cv_scores.mean())
print("CV Std :", cv_scores.std())

In [ ]:
# =====================================================
# STEP 8 : HYPERPARAMETER TUNING
# =====================================================

param_grid = {
    "max_depth":[3,5,7,10,None],
    "min_samples_split":[2,5,10,20],
    "min_samples_leaf":[1,2,5,10],
    "max_features":[None,"sqrt","log2"]}

grid = GridSearchCV(DecisionTreeRegressor(random_state=42),
    param_grid,cv=5,scoring="r2",n_jobs=-1)
grid.fit(X_train,y_train)
print(grid.best_params_)
best_model = grid.best_estimator_

# =====================================================
# STEP 8.5 : RANDOMIZED SEARCH
# =====================================================

random_search = RandomizedSearchCV(DecisionTreeRegressor(random_state=42),
    param_grid,n_iter=15,cv=5,scoring="r2",random_state=42,n_jobs=-1)
random_search.fit(X_train,y_train)
print(random_search.best_params_)


In [ ]:
# =====================================================
# STEP 9 : VALIDATION RE-EVALUATION with best model
# =====================================================

y_val_pred_best = best_model.predict(X_val)
val_r2 = r2_score(y_val,y_val_pred_best)

In [ ]:
# =====================================================
# STEP 10 : TRAIN VS VALIDATION with best model
# =====================================================

y_train_pred_best = best_model.predict(X_train)
train_r2 = r2_score(y_train,y_train_pred_best)
print("Train R2:",train_r2)
print("Validation R2:",val_r2)
# Interview: Train R2 = 1 but Validation R2 low means Tree Overfitting

In [ ]:
# =====================================================
# STEP 11 : TREE ANALYSIS
# =====================================================

# Tree Depth and leaf nodes

print("\nTree Depth:",best_model.get_depth())
print("Leaf Nodes:",best_model.get_n_leaves())

# Feature Importance

importance_df = pd.DataFrame({
    "Feature":X.columns,
    "Importance":best_model.feature_importances_}).sort_values(by="Importance",ascending=False)

plt.figure(figsize=(8,5))
plt.barh(importance_df["Feature"],importance_df["Importance"])
plt.title("Feature Importance")
plt.show()

# Max Depth Sensitivity

depths = [2,3,4,5,7,10,15,20]
scores = []

for depth in depths:
    model = DecisionTreeRegressor(max_depth=depth,random_state=42)
    model.fit(X_train,y_train)
    pred = model.predict(X_val)
    score = r2_score(y_val,pred)
    scores.append(score)

plt.plot(depths,scores,marker="o")
plt.xlabel("Depth")
plt.ylabel("Validation R2")
plt.title("Depth Sensitivity")
plt.show()

# Cost Complexity Pruning

path = best_model.cost_complexity_pruning_path(X_train,y_train)
alphas = path.ccp_alphas
alpha_scores = []

for alpha in alphas[:50]:
    model = DecisionTreeRegressor(ccp_alpha=alpha,random_state=42)
    model.fit(X_train,y_train)
    pred = model.predict(X_val)
    score = r2_score(y_val,pred)
    alpha_scores.append(score)

plt.plot(alphas[:50],alpha_scores)
plt.title("Pruning Analysis")
plt.xlabel("ccp_alpha")
plt.ylabel("Validation R2")
plt.show()

# Tree Visualization

plt.figure(figsize=(20,10))
plot_tree(best_model,feature_names=X.columns,filled=True,max_depth=3)
plt.show()


In [ ]:
# =====================================================
# STEP 12 : FINAL MODEL
# =====================================================

final_model = best_model

In [ ]:
# =====================================================
# STEP 13 : TEST EVALUATION
# =====================================================

y_test_pred = final_model.predict(X_test)
test_mae = mean_absolute_error(y_test,y_test_pred)
test_rmse = np.sqrt(mean_squared_error(y_test,y_test_pred))
test_r2 = r2_score(y_test,y_test_pred)

print("\nTEST RESULTS")
print("MAE :", test_mae)
print("RMSE:", test_rmse)
print("R2 :", test_r2)

In [ ]:
# =====================================================
# STEP 14 : INTERPRETATION
# =====================================================

print("\nTop Features")
print(importance_df.head(10))

In [ ]:
# =====================================================
# STEP 15 : DEPLOYMENT READINESS
# =====================================================

print("\nDEPLOYMENT")
print("CV Mean:",cv_scores.mean())
print("Validation R2:",val_r2)
print("Test R2:",test_r2)

In [ ]:
# =====================================================
# INTERVIEW QUESTIONS
# =====================================================

"""
1. How does Decision Tree Regressor work?
2. Why scaling is not required?
3. What is max_depth?
4. What happens if max_depth increases?
5. What is overfitting in trees?
6. What is pruning?
7. What is ccp_alpha?
8. What is feature importance?
9. What is min_samples_split?
10. What is min_samples_leaf?
11. Why are trees unstable?
12. What is variance in trees?
13. Why do trees overfit easily?
14. GridSearchCV vs RandomizedSearchCV?
15. Difference between Linear Regression and Tree Regression?
16. What is leaf node?
17. What is internal node?
18. What is root node?
19. How does a split happen?
20. Why Random Forest improves trees?
"""